# Lab 4: Agentic Integration and Evaluation
## Produce a Governed Monday Brief

**Timebox:** 45 minutes for integration, then 30 minutes for testing and evaluation.

This lab models an agent as a constrained workflow that calls small tools and stops at an approval gate. It does not send messages, change records, or make protected determinations.


## Mission

Combine market prioritization and grounded research into a fictional Monday Recruiting Operations Brief. The workflow must reject out-of-scope requests, cite its sources, surface data limitations, and require human approval.


In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

SCENARIO_DATE = pd.Timestamp('2026-08-14')
root_candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next((path for path in root_candidates if (path / 'data' / 'agentic').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Run from the repository root or notebooks directory.')
ARTIFACT_DIR = ROOT / 'artifacts' / 'agentic'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
markets = pd.read_csv(ROOT / 'data' / 'recommender' / 'market_inputs.csv')
documents = pd.read_csv(ROOT / 'data' / 'rag' / 'approved_documents.csv')
with open(ROOT / 'data' / 'agentic' / 'evaluation_cases.json') as handle:
    evaluation_cases = json.load(handle)


In [ ]:
STOPWORDS = {'a', 'an', 'and', 'for', 'from', 'in', 'is', 'of', 'on', 'or', 'the', 'to', 'with'}

def tokenize(text):
    return {token for token in re.findall(r'[a-z0-9]+', str(text).lower()) if token not in STOPWORDS}

def lexical_score(query, text):
    return len(tokenize(query) & tokenize(text)) / max(len(tokenize(query)), 1)


## Task 1: Enforce the decision boundary

Return OUT_OF_SCOPE if a request asks for eligibility, medical, waiver, conduct, security, or final assignment determination. Otherwise return NEEDS_HUMAN_APPROVAL. This is a simple policy gate, not a substitute for policy review.


In [ ]:
def classify_request(request):
    # TODO: Implement the allowed-scope gate.
    raise NotImplementedError('Complete Task 1 before running this cell.')


In [ ]:
assert classify_request('Rank markets for analyst review.') == 'NEEDS_HUMAN_APPROVAL'
assert classify_request('Determine medical eligibility for a prospect.') == 'OUT_OF_SCOPE'
print('Task 1 checks passed.')


## Task 2: Build the two tools

Create one tool that returns the top three eligible markets using a transparent score and one tool that returns the top two dated documents for a query. Both tools should be inspectable functions, not hidden behavior.


In [ ]:
def recommend_markets(markets):
    # TODO: Filter to approved and current markets, score them, and return the top three rows.
    raise NotImplementedError('Complete Task 2 before running this cell.')

def retrieve_evidence(query, documents, scenario_date):
    # TODO: Filter to effective documents, score retrieval text, and return the top two rows.
    raise NotImplementedError('Complete Task 2 before running this cell.')


In [ ]:
top_markets = recommend_markets(markets)
top_evidence = retrieve_evidence('stale capacity snapshot', documents, SCENARIO_DATE)
assert len(top_markets) == 3
assert top_markets['region_id'].iloc[0] == 'M-01'
assert top_evidence['doc_id'].iloc[0] == 'D-007'
print('Task 2 checks passed.')


## Task 3: Orchestrate a governed draft

For an allowed request, call both tools and return a dictionary with status, recommendations, evidence, citations, warnings, and approval_required. For an out-of-scope request, return only a status and an explanation. The allowed path must always end in NEEDS_HUMAN_APPROVAL.


In [ ]:
def run_workflow(request, markets, documents, scenario_date):
    # TODO: Route through classify_request, then use the two tools for allowed requests.
    raise NotImplementedError('Complete Task 3 before running this cell.')


In [ ]:
brief = run_workflow('Prepare a Monday market review for the current opportunity snapshot.', markets, documents, SCENARIO_DATE)
assert brief['status'] == 'NEEDS_HUMAN_APPROVAL'
assert brief['approval_required']
assert brief['citations']
assert len(brief['recommendations']) == 3
display(pd.DataFrame(brief['recommendations']))
print('Task 3 checks passed.')


## Task 4: Evaluate the workflow

Run each evaluation case and report whether the status matches, citations are present when required, and approval is required for allowed requests. Reliability is a test result, not an impression.


In [ ]:
def evaluate_workflow(cases, markets, documents, scenario_date):
    # TODO: Run the workflow for each case and return one evaluation row per case.
    raise NotImplementedError('Complete Task 4 before running this cell.')


In [ ]:
results = evaluate_workflow(evaluation_cases, markets, documents, SCENARIO_DATE)
assert results['passed'].all()
display(results)
print('Task 4 checks passed.')


## Handoff and debrief

The brief is a draft for a responsible leader. It is not an autonomous tasking system.


In [ ]:
with open(ARTIFACT_DIR / 'monday_brief.json', 'w') as handle:
    json.dump(brief, handle, indent=2)
results.to_csv(ARTIFACT_DIR / 'workflow_evaluation.csv', index=False)
print(f'Wrote agentic artifacts to: {ARTIFACT_DIR}')


1. Which tool output should a leader inspect before accepting the brief?
2. What failure case is missing from the current evaluation set?
3. What changes when a language model, rather than deterministic routing, selects tools?

### Optional extension

Add a repeated-run stability check after introducing a model-driven planner. Keep the approval gate unchanged.
